# Study 902 — Multi-Factor Composite 🧩

**"Single factors take turns working — so blend them." Does the blend beat the market?**

The practitioner's pitch for a multi-factor sleeve is *diversification*: value, quality,
momentum, min-vol and size each spend years out of favour, but a blend smooths the ride and
— the sell-side deck promises — earns a market-beating risk-adjusted return with less
factor-timing risk. We build the live version — an **equal-weight sleeve of VLUE + QUAL +
MTUM + USMV + SIZE**, rebalanced monthly — and race it against **SPY** on the
**excess-of-cash Sharpe** (both legs minus the BIL T-bill ETF), net of the rebalancing
turnover it actually pays (2013-08-31 → 2026-06-30, 155 months common to all
five sleeves).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint
`8b05ab0c64b8`); the live cells run the fast synthetic control. These five are the
flagship survivors of the 2010s smart-beta wave — the panel flatters the average factor-ETF
experience; named on the Signal axis.*


## 1. The idea in one picture

One factor is a bumpy ride: value spent 2017–2020 in the wilderness, momentum crashed in 2009, min-vol lags in melt-ups. Put all five in one basket, rebalance monthly, and the bad years of one are cushioned by the good years of another. That part is **real and mechanical** — a diversified blend has lower volatility than the average single sleeve. The open question is the *second* promise: does the smoother ride also **beat the market** once you net out costs?

In [1]:
R = dict(comp_sharpe=0.841, spy_sharpe=0.874, adv=-0.033, active_bps=-6.5, t_active=-0.73,
         comp_vol=14.1, spy_vol=14.5, mean_single_vol=15.2, mean_single_sharpe=0.786)
print('composite excess Sharpe : %.3f' % R['comp_sharpe'])
print('SPY excess Sharpe        : %.3f' % R['spy_sharpe'])
print('advantage (comp - SPY)   : %+.3f  (active %+.1f bps/mo, NW t %+.2f)'
      % (R['adv'], R['active_bps'], R['t_active']))
print('--- the diversification that DOES show up ---')
print('composite vol %.1f%% vs mean single sleeve %.1f%%' % (R['comp_vol'], R['mean_single_vol']))
print('composite Sharpe %.3f vs mean single sleeve %.3f' % (R['comp_sharpe'], R['mean_single_sharpe']))

composite excess Sharpe : 0.841
SPY excess Sharpe        : 0.874
advantage (comp - SPY)   : -0.033  (active -6.5 bps/mo, NW t -0.73)
--- the diversification that DOES show up ---
composite vol 14.1% vs mean single sleeve 15.2%
composite Sharpe 0.841 vs mean single sleeve 0.786


## 2. Is the machinery honest? A live synthetic control

Before trusting the race on the real tape, we prove the detector on a seeded toy world: five members that share a market and each carry an independent style factor, plus a benchmark and cash. Plant a per-annum blend edge (`edge=+3%`) and the Sharpe-advantage *t* must light up; set it to zero and the *t* must stay quiet. No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from multi_factor import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(n_months=168, edge_ann=0.0, seed=902))
planted = st.synthetic_detect(data.synthetic_world(n_months=168, edge_ann=0.03, seed=902))
print('null world   : Sharpe adv %+.3f  active NW t = %+.2f  (should be ~0)'
      % (null['sharpe_adv'], null['t_active_nw']))
print('planted world: Sharpe adv %+.3f  active NW t = %+.2f  (should light up)'
      % (planted['sharpe_adv'], planted['t_active_nw']))

null world   : Sharpe adv +0.007  active NW t = +0.21  (should be ~0)
planted world: Sharpe adv +0.198  active NW t = +3.66  (should light up)


## 3. The honest verdict — a real diversifier, not a market-beater

On the live tape the equal-weight sleeve earns an excess-of-cash Sharpe of **0.841** against SPY's **0.874** — an advantage of **-0.033**, i.e. it lands *just short* of the market. The active return is **-6.5 bps/mo** at NW *t* = **-0.73** (indistinguishable from zero, and the wrong side of it), a paired bootstrap puts the advantage CI at **[-0.205, +0.077]** straddling zero (P(adv<0)=0.80), and it **flips sign across the two eras** (+0.078 early → -0.085 late). Rebalancing costs are a **non-issue** — the sleeve turns over just 1.2% of NAV/mo (0.3 bps/yr) — so this isn't a cost story; there is simply **no market-beating edge** to cost. What *is* real: the blend's vol (14.1%) sits below the average single sleeve's (15.2%) and its Sharpe (0.841) beats the average single sleeve (0.786) — genuine diversification of *factor-timing* risk, just not of *market* risk. **Signal: Weak** (the diversification is real, the SPY-beating claim is not), **Tradability: Fragile** (cheap to hold, but what you hold is a marginally-lower-Sharpe, lower-vol version of the market).